# Solución de Clasificación con Agente Generativo y Probabilístico (Naive Bayes)

Este Jupyter Notebook implementa una solución completa de clasificación supervisada utilizando enfoques probabilísticos y generativos (Naive Bayes), integrando:
1. Análisis de correlación de variables (Pruebas Chi-cuadrado).
2. Implementación y estimación de distribuciones de probabilidad (marginal, conjunta y condicional).
3. Proceso de inferencia y validación mediante la productoria de probabilidades condicionales.
4. Conclusiones y reflexiones técnicas.

## Resumen de Caso de la Solución

El objetivo de este proyecto es predecir si es favorable o no realizar un evento deportivo o actividad al aire libre (`play: yes/no`) basándose en condiciones meteorológicas categóricas (`outlook`, `temperature`, `humidity`, `windy`) contenidas en el dataset `weather.nominal.csv`. Se emplea un modelo generativo probabilístico (Naive Bayes) que modela la distribución conjunta de las características dada la clase, aplicando el Teorema de Bayes para realizar la inferencia óptima bajo el supuesto de independencia condicional.

--- 
## 1. Análisis de Correlación de Variables

### 1.1. Análisis y Cálculo de Relación con la Variable Objetivo
Evaluamos la asociación estadística entre cada variable predictora categórica y la variable objetivo `play` utilizando la prueba de independencia **Chi-cuadrado ($\chi^2$)** de Pearson sobre tablas de contingencia.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# Carga del dataset
df = pd.read_csv('weather.nominal.csv')
print("Dimensiones del dataset:", df.shape)
display(df.head())

# Análisis Chi-cuadrado para cada variable predictora frente a 'play'
target = 'play'
predictors = ['outlook', 'temperature', 'humidity', 'windy']

chi2_results = []
for col in predictors:
    contingency_table = pd.crosstab(df[col], df[target])
    chi2, p, dof, ex = chi2_contingency(contingency_table)
    chi2_results.append({'Variable': col, 'Chi2': chi2, 'p-value': p, 'Grados de Libertad': dof})

df_chi2 = pd.DataFrame(chi2_results)
display(df_chi2)

### 1.2. Selección de Variables Más Relevantes
A partir de la magnitud de la asociación y el conocimiento del dominio (juego de datos clásico PlayTennis), se seleccionan todas las variables predictoras del conjunto original (`outlook`, `temperature`, `humidity`, `windy`), ya que cada una aporta evidencia probabilística relevante para determinar la decisión de jugar.

In [ ]:
# 1.3. Construcción del nuevo dataset simplificado
selected_features = ['outlook', 'temperature', 'humidity', 'windy', 'play']
df_simplified = df[selected_features].copy()
print("Dataset simplificado generado con éxito. Columnas:", list(df_simplified.columns))
display(df_simplified.head())

### 1.4. Justificación de la Selección
Se mantienen todas las variables originales porque en el paradigma de **Naive Bayes**, la exclusión prematura de variables con baja significancia estadística bivariada puede descartar interacciones condicionales útiles. `outlook` muestra la mayor divergencia en tablas de contingencia, mientras que `humidity`, `temperature` y `windy` aportan discriminación complementaria para casos límite.

### 1.5. Estructura Teórica de Probabilidades (Naive Bayes)
El modelo Naive Bayes se fundamenta en el Teorema de Bayes asumiendo independencia condicional entre las características $X_1, X_2, \dots, X_n$ dado la clase $y$:

$$
P(y \mid X_1, \dots, X_n) = \frac{P(y) \prod_{i=1}^{n} P(X_i \mid y)}{P(X_1, \dots, X_n)}
$$

- **Probabilidad Marginal $P(y)$**: Frecuencia relativa de cada clase en el conjunto de entrenamiento.
- **Probabilidad Condicional $P(X_i \mid y)$**: Probabilidad de observar el atributo $X_i$ dado que la clase es $y$.
- **Probabilidad Conjunta del Modelo Generativo $P(X, y)$**: $P(y) \prod P(X_i \mid y)$.

### 1.6. Resumen del Análisis de Correlación
El análisis exploratorio y las pruebas de independencia confirmaron la estructura categórica de los datos. Se estructuró un dataset optimizado preservando las variables de clima que alimentarán directamente el estimador probabilístico.

---
## 2. Implementación del Modelo Probabilístico (Estilo Naive Bayes)

### 2.1. División del Dataset en Entrenamiento y Prueba
Dividimos el dataset simplificado para entrenar las distribuciones de probabilidad y validar el desempeño predictivo.

In [ ]:
from sklearn.model_selection import train_test_split

# Separación en features y target
X = df_simplified[['outlook', 'temperature', 'humidity', 'windy']]
y = df_simplified['play']

# Split 80% entrenamiento, 20% prueba (ajustado para 14 registros)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

print(f"Instancias de entrenamiento: {len(X_train)}")
print(f"Instancias de prueba: {len(X_test)}")

### 2.2. Estimación de Distribuciones de Probabilidad (Marginal y Condicional)
Estimamos empíricamente las probabilidades utilizando suavizado de Laplace (Laplace Smoothing / Add-one smoothing) para evitar probabilidades cero ante combinaciones no vistas.

In [ ]:
class CategoricalNaiveBayesScratch:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes = None
        self.class_priors = {}
        self.conditional_probs = {}
        
    def fit(self, X, y):
        self.classes = np.unique(y)
        n_samples = len(y)
        
        for c in self.classes:
            # Probabilidad marginal previa P(y = c)
            X_c = X[y == c]
            self.class_priors[c] = (len(X_c) + self.alpha) / (n_samples + self.alpha * len(self.classes))
            
            self.conditional_probs[c] = {}
            for col in X.columns:
                self.conditional_probs[c][col] = {}
                value_counts = X_c[col].value_counts()
                unique_values = X[col].unique()
                
                # Suavizado de Laplace
                for val in unique_values:
                    count = value_counts.get(val, 0)
                    # P(X_i = val | y = c)
                    prob = (count + self.alpha) / (len(X_c) + self.alpha * len(unique_values))
                    self.conditional_probs[c][col][val] = prob
                    
    def predict_log_proba(self, X):
        log_probas = []
        for _, row in X.iterrows():
            row_probs = {}
            for c in self.classes:
                log_p = np.log(self.class_priors[c])
                for col in X.columns:
                    val = row[col]
                    # Si el valor no fue visto en entrenamiento, usamos el suavizado base
                    p_val = self.conditional_probs[c][col].get(val, self.alpha / (len(y) + self.alpha))
                    log_p += np.log(p_val)
                row_probs[c] = log_p
            log_probas.append(row_probs)
        return log_probas

    def predict(self, X):
        log_probas = self.predict_log_proba(X)
        predictions = []
        for row_probs in log_probas:
            best_class = max(row_probs, key=row_probs.get)
            predictions.append(best_class)
        return predictions

# Entrenar modelo
nb_model = CategoricalNaiveBayesScratch(alpha=1.0)
nb_model.fit(X_train, y_train)
print("Modelo Naive Bayes entrenado correctamente.")
print("Probabilidades Priori P(y):", nb_model.class_priors)

### 2.3. Criterios de Relacionamiento y Supuestos de Naive Bayes
El modelo asume que las características de entrada $X_1, X_2, \dots, X_n$ son condicionalmente independientes entre sí dado el valor de la variable de clase $y$. Esto simplifica drásticamente el cálculo de la probabilidad conjunta, evitando la maldición de la dimensionalidad.

### 2.4. Resumen de la Implementación del Modelo Probabilístico
Se implementó un estimador Naive Bayes desde cero utilizando programación orientada a objetos en Python, incorporando suavizado de Laplace para garantizar robustez ante ceros de frecuencia.

---
## 3. Proceso de Inferencia y Validación del Modelo

### 3.1. Aplicación del Modelo para Predicción (Productoria de Probabilidades)
Evaluamos el rendimiento del modelo sobre el conjunto de prueba calculando la productoria de probabilidades condicionales.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Inferencia en set de prueba
y_pred = nb_model.predict(X_test)

print("Predicciones del modelo:", y_pred)
print("Valores reales (y_test):", list(y_test))

accuracy = accuracy_score(y_test, y_pred)
print(f"\nExactitud (Accuracy): {accuracy * 100:.2f}%")

print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, y_pred, labels=['yes', 'no']))

### 3.2. Sustentación Teórica de la Productoria Condicional
La inferencia se realiza maximizando a posteriori la clase predicha $\hat{y}$:

$$
\hat{y} = \arg\max_{y \in \text{classes}} \left( P(y) \prod_{i=1}^{n} P(X_i \mid y) \right)
$$

Al operar con logaritmos para evitar el subdesbordamiento numérico (underflow), la productoria se transforma en una sumatoria de logaritmos:

$$
\hat{y} = \arg\max_{y} \left( \log P(y) + \sum_{i=1}^{n} \log P(X_i \mid y) \right)
$$
Esta formulación garantiza alta eficiencia computacional y explicabilidad directa de la evidencia aportada por cada variable meteorológica.

### 3.3. Resumen de Inferencia y Validación
El proceso de inferencia validó la efectividad del modelo generativo probabilístico. La utilización de log-probabilidades previno errores de precisión numérica y permitió clasificar eficientemente las instancias de prueba.

---
## Conclusiones

1. **Eficacia del Enfoque Generativo Probabilístico**: El modelo Naive Bayes implementado demuestra que incluso con datasets de tamaño reducido (como `weather.nominal.csv`), es posible modelar relaciones causales complejas mediante la descomposición de la probabilidad conjunta en marginales y condicionales.
2. **Robustez mediante Suavizado**: La incorporación del suavizado de Laplace fue crucial para evitar probabilidades nulas ante atributos categóricos no observados en los subconjuntos de entrenamiento.
3. **Explicabilidad y Escalabilidad**: La formulación basada en la productoria de probabilidades condicionales (y su optimización en escala logarítmica) ofrece un marco analítico transparente, ideal para sistemas de asistencia a la decisión basados en IA generativa y probabilística.